In [ ]:
# ✅ Install dependencies
!pip install -q gradio pyngrok pandas openpyxl scikit-learn

# ✅ Imports
import pandas as pd
import numpy as np
import gradio as gr
from pyngrok import ngrok
from google.colab import userdata
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ✅ Load Datasets
dataset1 = pd.read_csv("Dataset1.csv")     # Q&A
dataset2 = pd.read_csv("Dataset2.csv")     # User Profiles
dataset3 = pd.read_excel("Dataset3.xlsx")  # MI Scores (Not used here)
dataset4 = pd.read_excel("Dataset4.xlsx")  # Job Descriptions

# ✅ Prepare Dataset
df = dataset2.copy()
df.columns = [col.strip() for col in df.columns]
job_col = [col for col in df.columns if "Job title" in col][0]
df = df.rename(columns={job_col: "Job_Title"})

df["combined_profile"] = (
    df["What are your interests?"].fillna("") + " " +
    df["What are your skills ? (Select multiple if necessary)"].fillna("") + " " +
    df["What was your course in UG?"].fillna("") + " " +
    df["What is your UG specialization? Major Subject (Eg; Mathematics)"].fillna("") + " " +
    df["Have you done masters after undergraduation? If yes, mention your field of masters.(Eg; Masters in Mathematics)"].fillna("") + " " +
    df["If yes, please specify your certificate course title."].fillna("")
)

df = df[df["Job_Title"].notnull()]
df["Job_Title"] = df["Job_Title"].str.strip().str.lower()

# ✅ Encode Labels
le = LabelEncoder()
df["Job_Label"] = le.fit_transform(df["Job_Title"])

# ✅ Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df["combined_profile"], df["Job_Label"], test_size=0.2, random_state=42
)

# ✅ ML Pipeline
model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=1000)),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])
model.fit(X_train, y_train)

# ✅ Career Prediction Function (No PDF)
def predict_career(
    name, gender, ug_course, ug_spec, major_subject, interests, skills,
    cgpa, has_cert, cert_titles, is_working, job_title, has_masters, masters_spec, preferred_industries
):
    # Combine input
    combined = " ".join([
        interests or "", skills or "", ug_course or "", ug_spec or "",
        major_subject or "", masters_spec or "", cert_titles or ""
    ])

    # Predict
    probs = model.predict_proba([combined])[0]
    top_indices = np.argsort(probs)[::-1][:3]
    top_jobs = le.inverse_transform(top_indices)

    # Extract skills
    top_skills = [s.strip() for s in skills.split(",")] if skills else []
    skill_area_1 = top_skills[0] if len(top_skills) > 0 else "technology"
    skill_area_2 = top_skills[1] if len(top_skills) > 1 else "problem-solving"
    best_career = top_jobs[0].title()

    # 📝 Generate Advice
    output_text = f"""{name}, you are a very talented student with immense potential. Based on your skills, interests, and current learning path, you seem to be naturally inclined towards a career that involves critical thinking, creativity, and problem-solving.

A career in **{best_career}** could be an excellent fit for you. This path not only aligns with your strengths but also opens up a future full of exciting challenges and rewarding opportunities. You’ve shown a strong aptitude for **{skill_area_1}** and **{skill_area_2}**, which are in high demand in this field.

Stay curious, keep building your skill set, and never underestimate your ability to achieve great things. The journey ahead might have its ups and downs, but with your passion and consistency, success is inevitable. 🌟
"""

    # 📄 Add job description
    desc_match = dataset4[dataset4.iloc[:, 1].str.contains(best_career, case=False, na=False)]
    if not desc_match.empty:
        output_text += f"\n📄 **About {best_career}**: {desc_match.iloc[0, 2].strip()}\n"

    # 💬 Add sample Q&A
    qna_match = dataset1[dataset1["role"].str.contains(best_career, case=False, na=False)]
    if not qna_match.empty:
        sample = qna_match.iloc[:2]
        for _, row in sample.iterrows():
            output_text += f"\n💬 Q: {row['question']}\n🟢 A: {row['answer']}\n"

    # 🔁 Alternative Careers
    if len(top_jobs) > 1:
        output_text += "\n🔎 You can also explore:\n"
        for alt_job in top_jobs[1:]:
            output_text += f"   • {alt_job.title()}\n"

    return output_text

# ✅ Gradio Input Fields
inputs = [
    gr.Textbox(label="1. Full Name"),
    gr.Radio(["Male", "Female", "Other"], label="2. Gender"),
    gr.Textbox(label="3. UG Degree"),
    gr.Textbox(label="4. UG Specialization"),
    gr.Textbox(label="5. Major Subject"),
    gr.Textbox(label="6. Interests"),
    gr.Textbox(label="7. Skills (comma separated)"),
    gr.Textbox(label="8. CGPA or Percentage"),
    gr.Radio(["Yes", "No"], label="9. Any Certification Courses?"),
    gr.Textbox(label="10. Certification Titles"),
    gr.Radio(["Yes", "No"], label="11. Are you working?"),
    gr.Textbox(label="12. Job Title (If any)"),
    gr.Radio(["Yes", "No"], label="13. Have you done Masters?"),
    gr.Textbox(label="14. Master’s Specialization"),
    gr.Textbox(label="15. Preferred Careers/Industries")
]

# ✅ Gradio Interface (Single Text Output)
iface = gr.Interface(
    fn=predict_career,
    inputs=inputs,
    outputs=gr.Textbox(label="📝 Career Advice"),
    title="🎯 AI Career Advisor (ML-Powered)",
    description="Answer a few questions and receive personalized, motivational career advice based on your profile."
)

# ✅ Launch Gradio App (Non-blocking)
_, local_url, _ = iface.launch(share=False, prevent_thread_lock=True)

# ✅ Expose Public URL via ngrok
# Kill any existing ngrok processes
os.system("kill $(ps aux | grep 'ngrok' | grep -v 'grep' | awk '{print $2}')")

ngrok.set_auth_token(userdata.get("NGROK_AUTH_TOKEN"))  # 🔐 Set in Colab secrets
port = int(local_url.split(":")[-1].rstrip("/"))
public_url = ngrok.connect(port)

print("🔗 Public ngrok URL:", public_url)